# August Test Beam: Pedestal Subtraction and Calorimeter Calibration

Student notebook

> **Warning:** This notebook is for learning and cross-checking the August calibration workflow. The run lists, fits, and constants shown here may **not** be the final official calibration files. Always check with the latest logbook/configuration before using constants for final analysis.

> **Final-night data warning:** The final logbook contains later calibration additions, but the corresponding ROOT files are not ready in this notebook directory yet. In particular, final-night CAL0-on-new-channel, CAL7, and CAL18 runs are listed separately below and are **not** used by the fitting cells until their `.root` files are converted and added.

This notebook uses the August BL4S calorimeter calibration ROOT files. The goal is to understand:

- why pedestal subtraction is needed,
- how a PMT/calorimeter signal becomes a QDC peak,
- how we fit the peak,
- how we convert QDC counts to energy,
- how to check if the beam was centered on the selected calorimeter.

Work through the questions. Do not just run all cells: inspect the plots and explain what you see.

## 1. Experimental Setup

The calorimeter stack used for the August calibration was:

```text
CAL10 | CAL5  | CAL1  | CAL12
CAL17 | CAL4  | CAL18 | CAL7
CAL0  | CAL2  | CAL14 | CAL11
CAL8  | CAL19 | CAL9  | CAL13
```

Each calorimeter is read out by a QDC channel. The calibration runs were taken at 1, 2, and 3 GeV.

### Questions

1. Why do we need three beam energies instead of only one?
2. Why is it useful to know the physical neighbor cells?
3. Which calorimeters are missing a full calibration in this data set?

In [ ]:
from pathlib import Path
import csv
import math
import json
from IPython.display import Image, display

DATA_DIR = Path(".")
CONFIG_DIR = Path("QDCConfig")  # Existing input/reference constants.
OUTPUT_CONFIG_DIR = Path("QDCConfig_August_preliminary")  # Generated by this notebook.

CAL_TO_QDC = {
    "Cal0": 0, "Cal1": 1, "Cal2": 2, "Cal4": 3,
    "Cal5": 4, "Cal7": 5, "Cal8": 6, "Cal9": 7,
    "Cal10": 8, "Cal11": 9, "Cal12": 10, "Cal13": 11,
    "Cal14": 12, "Cal17": 13, "Cal18": 14, "Cal19": 15,
}

LAYOUT = [
    ["Cal10", "Cal5", "Cal1", "Cal12"],
    ["Cal17", "Cal4", "Cal18", "Cal7"],
    ["Cal0", "Cal2", "Cal14", "Cal11"],
    ["Cal8", "Cal19", "Cal9", "Cal13"],
]

EXPECTED_CALIBRATION_RUNS = {}
with open("august_calibration_runs_from_logbook.csv", newline="") as handle:
    for row in csv.DictReader(handle):
        if row.get("file_exists") != "yes":
            continue
        cal = row["cal"]
        energy = int(row["energy_mev"])
        run = int(row["run"])
        EXPECTED_CALIBRATION_RUNS.setdefault(cal, {})[energy] = run

STUDENT_CALIBRATION_RUNS = {
    "Cal4":  {1000: None, 2000: None, 3000: None},
    "Cal8":  {1000: None, 2000: None, 3000: None},
    "Cal19": {1000: None, 2000: None, 3000: None},
    "Cal9":  {1000: None, 2000: None, 3000: None},
    "Cal13": {1000: None, 2000: None, 3000: None},
    "Cal11": {1000: None, 2000: None, 3000: None},
    "Cal14": {1000: None, 2000: None, 3000: None},
    "Cal2":  {1000: None, 2000: None, 3000: None},
    "Cal0":  {1000: None, 2000: None, 3000: None},
    "Cal10": {1000: None, 2000: None, 3000: None},
    "Cal5":  {1000: None, 2000: None, 3000: None},
    "Cal1":  {1000: None, 2000: None, 3000: None},
    "Cal12": {1000: None, 2000: None, 3000: None},
    "Cal17": {1000: None, 2000: None, 3000: None},
}

def check_calibration_runs(student_runs):
    all_correct = True
    for cal in EXPECTED_CALIBRATION_RUNS:
        for energy in (1000, 2000, 3000):
            expected = EXPECTED_CALIBRATION_RUNS[cal][energy]
            value = student_runs.get(cal, {}).get(energy)
            if value == expected:
                print(f"correct: {cal:5s} {energy:4d} MeV -> {value}")
            else:
                all_correct = False
                if value is None:
                    print(f"missing: {cal:5s} {energy:4d} MeV")
                else:
                    print(f"check again: {cal:5s} {energy:4d} MeV -> {value}")
    if all_correct:
        print("\\nAll calibration runs are correct. You can continue.")
    else:
        print("\\nSome entries are missing or not correct yet. Read the logbook again and update STUDENT_CALIBRATION_RUNS.")
    return all_correct

PEDESTALS = {}
for cal, qdc in CAL_TO_QDC.items():
    f = CONFIG_DIR / f"QDC0_ch{qdc}_calibration.json"
    if f.exists():
        with f.open() as handle:
            d = json.load(handle)
        PEDESTALS[cal] = float(d.get("pedestal", {}).get("value", 0.0))
    else:
        PEDESTALS[cal] = 0.0

print("Loaded", len(PEDESTALS), "pedestal constants")
print("CONFIG_DIR is", CONFIG_DIR.resolve())
print("OUTPUT_CONFIG_DIR is", OUTPUT_CONFIG_DIR.resolve())
print("This notebook reads existing pedestal constants from QDCConfig and can generate new preliminary outputs in QDCConfig_August_preliminary.")
print("Available ROOT files:", len(list(DATA_DIR.glob("*.root"))))

def read_csv_rows(path):
    with open(path, newline="") as handle:
        return list(csv.DictReader(handle))

def print_rows(rows, columns=None, max_rows=20):
    rows = list(rows)[:max_rows]
    if not rows:
        print("(no rows)")
        return
    if columns is None:
        columns = list(rows[0])
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    print("  ".join("-" * widths[c] for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

## 2. Read the Logbook and Fill the Calibration Runs

Before fitting anything, read the August logbook and find the 1, 2, and 3 GeV calibration runs for each calorimeter.

The current executable notebook uses only runs for which ROOT files are available locally. The final-night additions from the final logbook are shown in the next cell so you know what has been added but is not ready for fitting here yet.

### About `QDCConfig`

`CONFIG_DIR = Path("QDCConfig")` points to a folder that should already be provided with the notebook. It may contain older/intermediate calibration JSON files, pedestal plots, energy-fit plots, and calibration-curve plots.

The students do **not** create those files by hand. The notebook reads existing pedestal constants from `QDCConfig` if they are available.

The new files produced by this notebook are written to:

```python
OUTPUT_CONFIG_DIR = Path("QDCConfig_August_preliminary")
```

This is deliberate: the August calibration is not finalized yet, so we keep generated student/preliminary outputs separate from any existing `QDCConfig` products. Once the missing final-night ROOT files are converted, the logbook run choices are checked, and the fitted constants are approved, the same code can be used to produce the final `QDCConfig` outputs.

In [ ]:
final_status = read_csv_rows("august_final_logbook_added_runs_status.csv")
print_rows(final_status, ["cal", "energy_mev", "run", "root_file_ready", "status", "note"], max_rows=20)

Now paste the run numbers into the dictionary below. Keep the energies in MeV:


```text
1 GeV -> 1000
2 GeV -> 2000
3 GeV -> 3000
```

Then run the checker. Correct entries will print `correct`. Missing or wrong entries will ask you to check again.

In [ ]:
# Replace each None with the run number you read from the logbook.
STUDENT_CALIBRATION_RUNS = {
    "Cal4":  {1000: None, 2000: None, 3000: None},
    "Cal8":  {1000: None, 2000: None, 3000: None},
    "Cal19": {1000: None, 2000: None, 3000: None},
    "Cal9":  {1000: None, 2000: None, 3000: None},
    "Cal13": {1000: None, 2000: None, 3000: None},
    "Cal11": {1000: None, 2000: None, 3000: None},
    "Cal14": {1000: None, 2000: None, 3000: None},
    "Cal2":  {1000: None, 2000: None, 3000: None},
    "Cal0":  {1000: None, 2000: None, 3000: None},
    "Cal10": {1000: None, 2000: None, 3000: None},
    "Cal5":  {1000: None, 2000: None, 3000: None},
    "Cal1":  {1000: None, 2000: None, 3000: None},
    "Cal12": {1000: None, 2000: None, 3000: None},
    "Cal17": {1000: None, 2000: None, 3000: None},
}

In [ ]:
if check_calibration_runs(STUDENT_CALIBRATION_RUNS):
    CALIBRATION_RUNS = STUDENT_CALIBRATION_RUNS
else:
    CALIBRATION_RUNS = STUDENT_CALIBRATION_RUNS
    print("You can inspect your entries, but fix the run map before trusting the fits.")

## 3. Pedestal Subtraction

The QDC does not read zero when no real particle signal is present. Electronics offsets, cable pickup, and baseline shifts produce a nonzero pedestal.

For each event:

```text
signal = raw_QDC - pedestal
```

### Questions

1. What would happen to the calibration if we did not subtract the pedestal?
2. Why can different QDC channels have different pedestal values?
3. Why should pedestal runs be taken with the same electronics configuration as the data?

In [ ]:
for cal in sorted(PEDESTALS, key=lambda x: int(x[3:])):
    print(f"{cal:5s} QDC0_ch{CAL_TO_QDC[cal]:2d} pedestal = {PEDESTALS[cal]:8.3f}")

## 4. Fit One Calibration Peak

Choose one calorimeter and one energy. The example below uses CAL0 at 3 GeV.

### Student task

Change `cal` and `energy_mev` to inspect other runs.

In [ ]:
# This notebook expects PyROOT, which is available on the BL4S/TDAQ machine.
# If PyROOT is not available on your laptop, run this notebook on the DAQ machine
# or inside the same ROOT environment used for toROOT.
import ROOT
ROOT.gROOT.SetBatch(True)

def fit_qdc_peak(cal, energy_mev, require_fs=False, fit_half_width=350, threshold=100):
    run = CALIBRATION_RUNS[cal][energy_mev]
    qdc = CAL_TO_QDC[cal]
    pedestal = PEDESTALS[cal]
    filename = DATA_DIR / f"{run}.root"
    if not filename.exists():
        raise FileNotFoundError(filename)

    f = ROOT.TFile.Open(str(filename))
    tree = f.Get("RAWdata")
    if not tree:
        raise RuntimeError(f"No RAWdata tree in {filename}")

    hist = ROOT.TH1D(
        f"h_{cal}_{energy_mev}_{'fs' if require_fs else 'all'}",
        f"{cal} {energy_mev/1000:.0f} GeV;QDC - pedestal;Events",
        450, 0, 4500,
    )
    hist.SetDirectory(0)

    expr = f"QDC0_ch{qdc}-{pedestal}>>{hist.GetName()}"
    cut = f"QDC0_ch{qdc}-{pedestal}>{threshold}"
    if require_fs:
        cut = f"({cut}) && NTDC0_ch4_leading>0 && NTDC0_ch5_leading>0"

    tree.Draw(expr, cut, "goff")
    if hist.GetEntries() < 50:
        return {
            "cal": cal, "energy_mev": energy_mev, "run": run,
            "entries": hist.GetEntries(), "mean": None, "sigma": None,
            "mean_error": None, "sigma_error": None,
        }

    peak = hist.GetBinCenter(hist.GetMaximumBin())
    fit = ROOT.TF1(f"fit_{hist.GetName()}", "gaus", max(0, peak-fit_half_width), peak+fit_half_width)
    hist.Fit(fit, "RQ0")

    return {
        "cal": cal,
        "energy_mev": energy_mev,
        "run": run,
        "entries": hist.GetEntries(),
        "peak_bin": peak,
        "mean": fit.GetParameter(1),
        "sigma": abs(fit.GetParameter(2)),
        "mean_error": fit.GetParError(1),
        "sigma_error": fit.GetParError(2),
        "chi2_ndf": fit.GetChisquare() / fit.GetNDF() if fit.GetNDF() else None,
        "hist": hist,
        "fit": fit,
    }

def forced_zero_slope(points):
    # Fit E = slope * ADC with intercept forced to zero.
    good = [p for p in points if p["mean"]]
    numerator = sum(p["energy_mev"] * p["mean"] for p in good)
    denominator = sum(p["mean"] ** 2 for p in good)
    return numerator / denominator if denominator else None

def draw_fit_png(point, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    c = ROOT.TCanvas(f"c_{point['cal']}_{point['energy_mev']}", "calibration point", 900, 650)
    point["hist"].SetLineColor(ROOT.kBlue + 1)
    point["hist"].Draw("hist")
    point["fit"].SetLineColor(ROOT.kRed)
    point["fit"].Draw("same")
    c.SaveAs(str(output_dir / f"QDC0_ch{CAL_TO_QDC[point['cal']]}_energy_{point['energy_mev']}.png"))

def show_qdc_peak(cal, energy_mev, require_fs=False, output_dir=Path("notebook_plots")):
    output_dir.mkdir(parents=True, exist_ok=True)
    result = fit_qdc_peak(cal, energy_mev, require_fs=require_fs)
    for k, v in result.items():
        if k not in ("hist", "fit"):
            print(k, v)
    c = ROOT.TCanvas(f"c_peak_{cal}_{energy_mev}_{'fs' if require_fs else 'all'}", "QDC peak", 900, 650)
    result["hist"].SetLineColor(ROOT.kBlue + 1)
    result["hist"].Draw("hist")
    if result.get("fit"):
        result["fit"].SetLineColor(ROOT.kRed)
        result["fit"].Draw("same")
    output_png = output_dir / f"{cal}_{energy_mev}_{'fs' if require_fs else 'all'}_peak.png"
    c.SaveAs(str(output_png))
    display(Image(filename=str(output_png)))
    return result


def draw_calibration_curve(cal, points, slope, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    good = [p for p in points if p["mean"]]
    graph = ROOT.TGraphErrors(len(good))
    for i, p in enumerate(good):
        graph.SetPoint(i, p["mean"], p["energy_mev"])
        graph.SetPointError(i, p["mean_error"] or 0.0, 0.0)
    graph.SetTitle(f"{cal} calibration;Pedestal-subtracted QDC;Energy [MeV]")
    graph.SetMarkerStyle(20)
    graph.SetMarkerColor(ROOT.kBlue + 1)
    line = ROOT.TF1(f"line_{cal}", f"{slope}*x", 0, 4500)
    line.SetLineColor(ROOT.kRed)
    c = ROOT.TCanvas(f"c_curve_{cal}", "calibration curve", 900, 650)
    graph.Draw("AP")
    line.Draw("same")
    c.SaveAs(str(output_dir / f"QDC0_ch{CAL_TO_QDC[cal]}_calibration_curve.png"))

def save_calibration_outputs(cal, require_fs=False, output_dir=OUTPUT_CONFIG_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)
    points = [fit_qdc_peak(cal, e, require_fs=require_fs) for e in (1000, 2000, 3000)]
    good = [p for p in points if p["mean"]]
    if len(good) < 2:
        raise RuntimeError(f"Not enough valid points for {cal}")

    slope = forced_zero_slope(good)
    qdc = CAL_TO_QDC[cal]

    for p in good:
        draw_fit_png(p, output_dir)
    draw_calibration_curve(cal, good, slope, output_dir)

    payload = {
        "calorimeter": cal,
        "channel": f"QDC0_{qdc}",
        "selection": "FS0_FS1" if require_fs else "normal",
        "status": "preliminary",
        "note": "Generated by August_Calorimeter_Calibration notebook. Do not use as final constants until the run list, missing final-night ROOT files, and fits are approved.",
        "pedestal": {
            "value": PEDESTALS[cal],
            "source": str(CONFIG_DIR / f"QDC0_ch{qdc}_calibration.json"),
        },
        "calibration": {
            "offset": 0.0,
            "slope": slope,
            "model": "Energy_MeV = slope * (QDC - pedestal)",
            "forced_through_zero": True,
        },
        "calibration_points": [
            {
                "energy": p["energy_mev"],
                "run": p["run"],
                "file": str(DATA_DIR / f"{p['run']}.root"),
                "adc": p["mean"],
                "adc_uncertainty": p["sigma"],
                "mean_error": p["mean_error"],
                "entries": p["entries"],
                "chi2_ndf": p.get("chi2_ndf"),
            }
            for p in good
        ],
    }
    output_json = output_dir / f"QDC0_ch{qdc}_calibration.json"
    with output_json.open("w") as handle:
        json.dump(payload, handle, indent=2)
        handle.write("\\n")
    return payload

def generate_all_qdc_outputs(require_fs=False, output_dir=OUTPUT_CONFIG_DIR):
    generated = []
    skipped = []
    for cal in sorted(CALIBRATION_RUNS, key=lambda x: int(x[3:])):
        try:
            payload = save_calibration_outputs(cal, require_fs=require_fs, output_dir=output_dir)
            generated.append((cal, payload["channel"], payload["calibration"]["slope"]))
        except Exception as exc:
            skipped.append((cal, str(exc)))
    print(f"Generated {len(generated)} calibration JSON files in {output_dir}")
    for cal, channel, slope in generated:
        print(f"generated: {cal:5s} {channel:8s} slope = {slope:.6g} MeV/ADC")
    if skipped:
        print("\\nSkipped:")
        for cal, reason in skipped:
            print(f"{cal:5s}: {reason}")
    return generated, skipped

def compare_pedestal_runs(run_hv_on=1786828994, run_hv_off=1786829188):
    rows = []
    for qdc in range(16):
        values = {}
        for label, run in [("hv_on", run_hv_on), ("hv_off", run_hv_off)]:
            filename = DATA_DIR / f"{run}.root"
            if not filename.exists():
                values[label] = None
                continue
            f = ROOT.TFile.Open(str(filename))
            tree = f.Get("RAWdata")
            hist = ROOT.TH1D(f"h_ped_{label}_{qdc}", f"QDC0_ch{qdc} {label}", 500, 0, 500)
            hist.SetDirectory(0)
            tree.Draw(f"QDC0_ch{qdc}>>{hist.GetName()}", "", "goff")
            values[label] = {
                "entries": hist.GetEntries(),
                "mean": hist.GetMean(),
                "rms": hist.GetRMS(),
            }
        on = values["hv_on"]
        off = values["hv_off"]
        rows.append({
            "qdc_channel": f"QDC0_ch{qdc}",
            "hv_on_mean": None if on is None else on["mean"],
            "hv_off_mean": None if off is None else off["mean"],
            "mean_difference": None if on is None or off is None else on["mean"] - off["mean"],
            "hv_on_rms": None if on is None else on["rms"],
            "hv_off_rms": None if off is None else off["rms"],
            "hv_on_entries": None if on is None else on["entries"],
            "hv_off_entries": None if off is None else off["entries"],
        })
    return rows

## 4a. Compare Pedestal Runs With HV ON and HV OFF

The final logbook has pedestal runs with HV ON and HV OFF:

```text
HV ON:  1786828994
HV OFF: 1786829188
```

Run the comparison below and look at the mean and RMS of each QDC channel.

### Student task

1. Which channels change the most between HV ON and HV OFF?
2. Does HV mainly change the pedestal mean, the RMS/noise, or both?
3. If a channel has a much larger RMS with HV ON, what could that mean?

In [ ]:
pedestal_comparison = compare_pedestal_runs(run_hv_on=1786828994, run_hv_off=1786829188)
print_rows(
    pedestal_comparison,
    ["qdc_channel", "hv_on_mean", "hv_off_mean", "mean_difference", "hv_on_rms", "hv_off_rms"],
    max_rows=20,
)

## 4b. CAL0 Troubleshooting Discussion

The final logbook notes that CAL0 looked problematic, even without HV, and was later moved to a new channel/adaptor.

In the real debugging sequence, the team checked the signal with HV ON, with HV OFF, and after the delay units. The issue was eventually traced to a faulty QDC adaptor, so CAL0 was moved to Ch16.

### Questions

1. If CAL0 is noisy even with HV OFF, is the PMT the most likely problem?
2. What hardware pieces would you check first?
3. Why does moving CAL0 to a new QDC channel help diagnose the problem?
4. If the signal after the delay unit looks reasonable but the QDC readout is still abnormal, what does that suggest?
5. Why does moving CAL0 to Ch16 help?

In [ ]:
cal = "Cal0"
energy_mev = 3000
result = show_qdc_peak(cal, energy_mev, require_fs=False)

### Questions

1. What does the Gaussian mean represent physically?
2. What does the Gaussian sigma tell us?
3. Is the whole distribution Gaussian? If not, why do we fit only the peak region?

## 5. Build a Calibration Curve

For each calorimeter we fit the QDC peak at 1, 2, and 3 GeV. Then we force the calibration through zero:

```text
Energy [MeV] = slope * pedestal_subtracted_QDC
```

### Student task

Pick a calorimeter and compute its slope.

In [ ]:
cal = "Cal12"
points = [fit_qdc_peak(cal, e, require_fs=False) for e in (1000, 2000, 3000)]
for p in points:
    print(p["energy_mev"], "MeV:", "mean =", p["mean"], "sigma =", p["sigma"], "entries =", p["entries"])

slope = forced_zero_slope(points)
print("\nCalibration slope:", slope, "MeV / ADC")
print("Example: 1500 ADC ->", slope * 1500, "MeV")

### Questions

1. What assumption is made when we force the line through zero?
2. When might that assumption fail?
3. If the 1, 2, and 3 GeV points do not lie on a straight line, what could be wrong?

## 5b. Generate the QDCConfig Outputs

After your run map is correct, the notebook can generate calibration outputs automatically. This creates one JSON file and diagnostic PNG plots for each calorimeter.

The outputs are written to `QDCConfig_August_preliminary`, not the old `QDCConfig` folder. Treat them as preliminary until the calibration is approved.

In [ ]:
# Run this only after the checker says the calibration runs are correct.
generated, skipped = generate_all_qdc_outputs(require_fs=False, output_dir=OUTPUT_CONFIG_DIR)

## 6. Normal Trigger vs FS0&FS1 Selection

We can repeat the calibration using only events where both FS0 and FS1 have a TDC hit:

```text
NTDC0_ch4_leading > 0 and NTDC0_ch5_leading > 0
```

### Student task

Compare the same calorimeter with and without the FS0&FS1 cut.

In [ ]:
cal = "Cal12"
for require_fs in (False, True):
    label = "FS0&FS1 cut" if require_fs else "normal trigger"
    points = [fit_qdc_peak(cal, e, require_fs=require_fs) for e in (1000, 2000, 3000)]
    slope = forced_zero_slope(points)
    print("\n", label)
    print("slope =", slope)
    for p in points:
        print(p["energy_mev"], "mean =", p["mean"], "entries =", p["entries"])

### Questions

1. Why can the FS0&FS1-selected peak be slightly different from the normal-trigger peak?
2. If your physics analysis requires FS0&FS1, which calibration should you use?
3. Why did CAL4 fail with the FS0&FS1 selection in our files?

## 7. Beam Centering and Neighbor Checks

A calibration run should mainly illuminate the target calorimeter. If a neighbor has a larger signal than the target, the first suspicion is that something needs checking. It can be beam position, but it can also be a faulty neighbor readout, wrong channel mapping, timing/gate problem, HV/gain difference, or bad calibration constant.

The files `august_calibration_centering_summary.csv` and `august_calibration_neighbor_counts.csv` contain this check.

In [ ]:
summary = read_csv_rows("august_calibration_centering_summary.csv")
bad = [row for row in summary if row["status"] != "centered"]
print_rows(bad, ["target", "energy_mev", "run", "target_direct_neighbor_leading_fraction", "worst_neighbor", "status", "note"])

### Questions

1. Which calorimeter should be repeated first?
2. CAL5 looked strange because CAL1 was large in the CAL5 runs. What checks would tell you whether this is beam position or a CAL1/readout problem?
3. CAL12 has lower QDC counts than in June. Does the neighbor check say it was badly centered?

## 8. PMT High Voltage and Detector Differences

The PMTs are not identical. The same light yield can produce different signal sizes depending on PMT gain, optical coupling, cable/electronics response, and high voltage.

### Questions

1. Why do different PMTs need different high voltages?
2. What happens if the PMT voltage is too low?
3. What happens if the PMT voltage is too high?
4. Why should we avoid changing HV after calibration?

## 9. Final Checklist

For each calorimeter, decide:

- Do we have 1, 2, and 3 GeV runs?
- Is the beam centered well enough?
- Are the Gaussian fits stable?
- Should we use normal-trigger or FS0&FS1 calibration?
- Does this calorimeter need to be repeated?